In [1]:
# ==========================================================
# CELLULE 1 - CONFIGURATION INTERACTIVE
# ==========================================================
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt  # type: ignore[reportMissingImports]
import numpy as np
import pandas as pd

try:
    from IPython.display import display  # type: ignore[reportMissingImports]
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

try:
    from ipywidgets import (  # type: ignore[reportMissingImports]
        Dropdown,
        IntSlider,
        IntText,
        Output,
        SelectMultiple,
        VBox,
        Label,
    )
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

try:
    from pynq import Overlay, allocate  # type: ignore[reportMissingImports]
    HAS_PYNQ = True
except Exception:
    HAS_PYNQ = False
    Overlay = None
    allocate = None

# Constantes
FS_DEFAULT = 11_999_000.0
IF_DEFAULT = 3_563_000.0
N_DEFAULT = 11999
NB_PHASES_DEFAULT = 1023
EPS = 1e-12
# === Constantes et LUTs pour acquisition compatible HLS ===
DDS_LUT_SIZE = 1024
DDS_PHASE_BITS = 32
DDS_LUT_BITS = 10

def load_dds_luts_from_header(header_file_path):
    """
    Parse les LUT DDS depuis le fichier .h.
    Retourne DDS_SIN_LUT et DDS_COS_LUT sous forme de tableaux numpy.
    """
    with open(header_file_path, 'r') as f:
        content = f.read()

    # Extraction DDS_SIN_LUT
    sin_match = re.search(r'static const osc_t DDS_SIN_LUT\[DDS_LUT_SIZE\] = \{(.*?)\};', content, re.DOTALL)
    if sin_match:
        sin_values = re.findall(r'\(osc_t\)([0-9e\.\-\+]+)', sin_match.group(1))
        DDS_SIN_LUT = np.array([float(v) for v in sin_values], dtype=np.float32)
    else:
        raise ValueError("DDS_SIN_LUT not found in header file")

    # Extraction DDS_COS_LUT
    cos_match = re.search(r'static const osc_t DDS_COS_LUT\[DDS_LUT_SIZE\] = \{(.*?)\};', content, re.DOTALL)
    if cos_match:
        cos_values = re.findall(r'\(osc_t\)([0-9e\.\-\+]+)', cos_match.group(1))
        DDS_COS_LUT = np.array([float(v) for v in cos_values], dtype=np.float32)
    else:
        raise ValueError("DDS_COS_LUT not found in header file")

    return DDS_SIN_LUT, DDS_COS_LUT

# Charger les LUTs depuis le fichier header (adapter le chemin si besoin)
DDS_SIN_LUT, DDS_COS_LUT = load_dds_luts_from_header('dds_lut_rom.h')

def hz_to_phase_inc(freq_hz):
    SCALE = 1 << DDS_PHASE_BITS
    num = freq_hz * SCALE
    if num >= 0:
        num += IF_DEFAULT // 2
    else:
        num -= IF_DEFAULT // 2
    return int(num // IF_DEFAULT)


HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)

METHOD_LABELS = {
    "cpu_float": "CPU float",
    "cpu_fpga_compatible": "CPU compatible FPGA",
    "fpga": "FPGA",
}

signals_dir = Path("signals")
prn_dir = Path("PRN")


def collect_signal_files_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    files = sorted(base_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            with file_path.open("r", encoding="utf-8", errors="ignore") as handle:
                first = handle.readline().strip()
            if HEADER_RE.search(first):
                valid_files.append(file_path.name)
        except Exception:
            continue
    return valid_files


def get_available_prns_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in base_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


available_signals = collect_signal_files_for_ui(signals_dir)
available_prns = get_available_prns_for_ui(prn_dir)


DEFAULT_CONFIG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "limit_signals": None,
    "run_methods": ["cpu_float", "cpu_fpga_compatible"] + (["fpga"] if HAS_PYNQ else []),
    "prn_list": None,
    "fs_hz": FS_DEFAULT,
    "if_hz": IF_DEFAULT,
    "n_samples": N_DEFAULT,
    "nb_phases": NB_PHASES_DEFAULT,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 500,
    "detection_ratio_min": 2.5,
    "winner_margin_db_min": 3.0,
    "doppler_tol_hz": 500,
    "phase_tol_chip": 3,
    "bit_path": "sadfsnr.bit",
    "prn_mode": "all_available",
    "export_prefix": "validation_gnss",
}


WIDGET_STATE = {
    "enabled": False,
    "limit_signals_mode": None,
    "limit_signals_value": None,
    "fd_mode": None,
    "fd_start_widget": None,
    "fd_end_widget": None,
    "fd_step_widget": None,
    "prn_mode": None,
    "prn_selector": None,
}


def get_current_config():
    cfg = dict(DEFAULT_CONFIG)
    if not WIDGET_STATE["enabled"]:
        return cfg

    limit_mode = WIDGET_STATE["limit_signals_mode"].value
    cfg["limit_signals"] = None if limit_mode == 0 else int(WIDGET_STATE["limit_signals_value"].value)

    fd_mode_value = WIDGET_STATE["fd_mode"].value
    if fd_mode_value == 0:
        cfg["fd_start"] = -10000
        cfg["fd_end"] = 10000
        cfg["fd_step"] = 250
    else:
        cfg["fd_start"] = int(WIDGET_STATE["fd_start_widget"].value)
        cfg["fd_end"] = int(WIDGET_STATE["fd_end_widget"].value)
        cfg["fd_step"] = int(WIDGET_STATE["fd_step_widget"].value)

    prn_mode_value = WIDGET_STATE["prn_mode"].value
    if prn_mode_value == 0:
        cfg["prn_mode"] = "all_available"
        cfg["prn_list"] = None
    else:
        cfg["prn_mode"] = "selected"
        cfg["prn_list"] = list(WIDGET_STATE["prn_selector"].value)

    return cfg


print("=== CONFIGURATION INTERACTIVE ===")
print(f"Signaux trouvés: {len(available_signals)}")
print(f"PRN disponibles: {available_prns}")
print()

if HAS_WIDGETS and available_signals and available_prns:
    limit_signals_mode = Dropdown(
        options=[("Tous les signaux", 0), ("Nombre précis", 1)],
        value=0,
        description="Signaux:",
    )
    limit_signals_value = IntText(
        value=min(5, len(available_signals)),
        description="Combien:",
    )
    limit_signals_value.layout.display = "none"

    fd_mode = Dropdown(
        options=[("Plage standard", 0), ("Plage personnalisée", 1)],
        value=0,
        description="Doppler:",
    )
    fd_start_widget = IntSlider(value=-10000, min=-50000, max=0, step=100, description="Fd début:")
    fd_end_widget = IntSlider(value=10000, min=0, max=50000, step=100, description="Fd fin:")
    fd_step_widget = IntSlider(value=500, min=100, max=5000, step=50, description="Fd step:")
    fd_start_widget.layout.display = "none"
    fd_end_widget.layout.display = "none"
    fd_step_widget.layout.display = "none"

    prn_mode = Dropdown(
        options=[("Tous les PRN", 0), ("PRN spécifiques", 1)],
        value=0,
        description="PRN:",
    )
    prn_selector = SelectMultiple(
        options=[(str(prn), prn) for prn in available_prns],
        value=tuple(available_prns[:1]) if available_prns else (),
        rows=min(10, len(available_prns)),
        description="Sélection:",
    )
    prn_selector.layout.display = "none"

    config_output = Output()

    def update_limit_display(change):
        limit_signals_value.layout.display = "flex" if change.new == 1 else "none"
        update_config_display()

    def update_doppler_display(change):
        is_custom = change.new == 1
        fd_start_widget.layout.display = "flex" if is_custom else "none"
        fd_end_widget.layout.display = "flex" if is_custom else "none"
        fd_step_widget.layout.display = "flex" if is_custom else "none"
        update_config_display()

    def update_prn_display(change):
        prn_selector.layout.display = "flex" if change.new == 1 else "none"
        update_config_display()

    def update_config_display(change=None):
        cfg = get_current_config()
        with config_output:
            config_output.clear_output()
            print("=== CONFIGURATION CHOISIE ===")
            n_signaux = len(available_signals) if cfg["limit_signals"] is None else cfg["limit_signals"]
            print(f"Nombre de signaux: {n_signaux}")
            print(f"Plage Doppler: [{cfg['fd_start']}, {cfg['fd_end']}] Hz")
            print(f"Pas Doppler fd_step: {cfg['fd_step']} Hz")
            if cfg["prn_list"] is None:
                print(f"PRN: tous ({len(available_prns)} disponibles)")
            else:
                print(f"PRN: {cfg['prn_list']}")

    limit_signals_mode.observe(update_limit_display, names="value")
    limit_signals_value.observe(update_config_display, names="value")
    fd_mode.observe(update_doppler_display, names="value")
    fd_start_widget.observe(update_config_display, names="value")
    fd_end_widget.observe(update_config_display, names="value")
    fd_step_widget.observe(update_config_display, names="value")
    prn_mode.observe(update_prn_display, names="value")
    prn_selector.observe(update_config_display, names="value")

    WIDGET_STATE.update({
        "enabled": True,
        "limit_signals_mode": limit_signals_mode,
        "limit_signals_value": limit_signals_value,
        "fd_mode": fd_mode,
        "fd_start_widget": fd_start_widget,
        "fd_end_widget": fd_end_widget,
        "fd_step_widget": fd_step_widget,
        "prn_mode": prn_mode,
        "prn_selector": prn_selector,
    })

    display(VBox([
        Label("=== SIGNAUX ==="),
        limit_signals_mode,
        limit_signals_value,
        Label("=== DOPPLER ==="),
        fd_mode,
        fd_start_widget,
        fd_end_widget,
        fd_step_widget,
        Label("=== PRN ==="),
        prn_mode,
        prn_selector,
        Label(""),
        config_output,
    ]))

    update_config_display()
else:
    if not HAS_WIDGETS:
        print("[INFO] ipywidgets non disponible: configuration non interactive.")
    elif not available_signals:
        print("[INFO] Aucun signal détecté dans signals.")
    elif not available_prns:
        print("[INFO] Aucun PRN détecté dans PRN.")

CONFIG = get_current_config()
print()
print(pd.Series(CONFIG, dtype="object"))
print(f"\nPYNQ: {HAS_PYNQ}")

=== CONFIGURATION INTERACTIVE ===
Signaux trouvés: 86
PRN disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]




signals_dir                                            signals
prn_dir                                                    PRN
signal_glob                                              *.csv
limit_signals                                             None
run_methods             [cpu_float, cpu_fpga_compatible, fpga]
prn_list                                                  None
fs_hz                                               11999000.0
if_hz                                                3563000.0
n_samples                                                11999
nb_phases                                                 1023
fd_start                                                -10000
fd_end                                                   10000
fd_step                                                    250
detection_ratio_min                                        2.5
winner_margin_db_min                                       3.0
doppler_tol_hz                                        

In [ ]:
# ==========================================================
# CELLULE 2 - CAMPAGNE DE VALIDATION CPU / FPGA
# Produit:
# - validation_gnss_raw_results.csv
# - validation_gnss_winner_results.csv
# - validation_gnss_domain_summary.csv
# ==========================================================
FPGA_CONTEXT = None


def to_s32(x):
    x = int(x)
    return x if x < 0x80000000 else x - 0x100000000


def phase_error_mod(measured, expected, modulo=1023):
    d = (int(measured) - int(expected)) % int(modulo)
    return int(min(d, modulo - d))


def parse_meta_from_header(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline().strip()
    match = HEADER_RE.search(first)
    if not match:
        raise ValueError(f"Métadonnées introuvables dans {csv_file.name}")
    return {
        "prn": int(match.group("prn")),
        "snr": float(match.group("snr")),
        "doppler": int(round(float(match.group("doppler")))),
        "ca_shift": int(match.group("ca")),
    }


def load_signal_pm1(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        line1 = handle.readline().strip()
        line2 = handle.readline().strip()
    if line1 and not line1.startswith("#") and not line2:
        line2 = line1
    if not line2:
        raise ValueError(f"Signal vide: {csv_file}")
    data = np.fromstring(line2, sep=",", dtype=np.float32)
    if data.size == 0:
        raise ValueError(f"Format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float32)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv",
        prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv",
        prn_dir / f"prn_{prn}.csv",
    ]
    prn_path = next((path for path in candidates if path.exists()), None)
    if prn_path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(prn_path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court ({seq.size}) dans {prn_path}, attendu >= {n_samples}")
    return seq[:n_samples].astype(np.float32)


def get_available_prns(prn_dir):
    prn_dir = Path(prn_dir)
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in prn_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


def collect_signal_files(signals_dir, signal_glob):
    signals_dir = Path(signals_dir)
    files = sorted(signals_dir.glob(signal_glob)) if signals_dir.exists() else []
    if not files and signals_dir.exists():
        files = sorted(signals_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            parse_meta_from_header(file_path)
            valid_files.append(file_path)
        except Exception:
            continue
    return valid_files


def _precompute_prn_shifts(prn, n_samples, nb_phases):
    shifts = np.floor(np.arange(nb_phases) * n_samples / nb_phases).astype(int)
    prn_shifted = np.empty((nb_phases, n_samples), dtype=np.float32)
    for tau in range(nb_phases):
        prn_shifted[tau] = np.roll(prn, -shifts[tau])
    return prn_shifted


def cpu_float_acquisition(signal, prn, cfg):
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fs_hz = float(cfg["fs_hz"])
    if_hz = float(cfg["if_hz"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])

    signal = np.asarray(signal[:n_samples], dtype=np.complex64)
    prn = np.asarray(prn[:n_samples], dtype=np.float32)
    prn_shifted = _precompute_prn_shifts(prn, n_samples, nb_phases)
    doppler_range = np.arange(fd_start, fd_end + 1, fd_step, dtype=np.int32)
    corr_matrix = np.zeros((len(doppler_range), nb_phases), dtype=np.float64)
    n = np.arange(n_samples, dtype=np.float64)
    ts = 1.0 / fs_hz

    t0 = time.perf_counter()
    for k, fd in enumerate(doppler_range):
        carrier = np.exp(-1j * 2.0 * np.pi * (if_hz + float(fd)) * n * ts)
        corr = prn_shifted @ (signal * carrier)
        corr_matrix[k, :] = np.abs(corr) ** 2
    t1 = time.perf_counter()

    row, col = np.unravel_index(int(np.argmax(corr_matrix)), corr_matrix.shape)
    peak = float(corr_matrix[row, col])
    mean_corr = float(np.mean(corr_matrix))
    peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
    return {
        "method": "cpu_float",
        "detected": int(peak_ratio >= ratio_min),
        "doppler": int(doppler_range[row]),
        "phase": int(col),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t1 - t0),
    }


def _ap_int2_wrap(x):
    x_int = np.rint(np.real(x)).astype(np.int64)
    x_int = x_int & 0x3
    return np.where(x_int >= 2, x_int - 4, x_int).astype(np.int8)

def cpu_fpga_compatible_acquisition(signal, prn, cfg):
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fs_hz = float(cfg["fs_hz"])
    if_hz = float(cfg["if_hz"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])

    signal = np.asarray(signal[:n_samples])
    prn = np.asarray(prn[:n_samples])
    ts = 1.0 / fs_hz
    n = np.arange(n_samples, dtype=np.float64)
    sig_q = _ap_int2_wrap(signal).astype(np.float32)
    prn_bit = (np.real(prn) > 0).astype(np.int8)
    prn_2n = np.empty(2 * n_samples, dtype=np.int8)
    prn_2n[:n_samples] = prn_bit
    prn_2n[n_samples:] = prn_bit
    tau_start = np.floor(np.arange(nb_phases) * n_samples / nb_phases).astype(np.int32)
    idx = tau_start[:, None] + np.arange(n_samples, dtype=np.int32)[None, :]
    local_mat = prn_2n[idx].astype(np.float32)

    doppler_range = np.arange(fd_start, fd_end + 1, fd_step, dtype=np.int32)
    corr_matrix = np.zeros((len(doppler_range), nb_phases), dtype=np.float64)

    t0 = time.perf_counter()
    for k, fd in enumerate(doppler_range):
        phase_inc = hz_to_phase_inc(-(IF_DEFAULT + fd))
        phase_acc = 0
        mixI_loc = np.zeros(n_samples, dtype=np.float32)
        mixQ_loc = np.zeros(n_samples, dtype=np.float32)
        for nn in range(n_samples):
            lut_idx = (phase_acc >> (DDS_PHASE_BITS - DDS_LUT_BITS)) & (DDS_LUT_SIZE - 1)
            c = DDS_COS_LUT[lut_idx]
            s = DDS_SIN_LUT[lut_idx]
            x = sig_q[nn]
            mixI_loc[nn] = x * c
            mixQ_loc[nn] = -x * s
            phase_acc = (phase_acc + phase_inc) & ((1 << DDS_PHASE_BITS) - 1)

        for tau in range(nb_phases):
            accI_even = 0.0
            accI_odd = 0.0
            accQ_even = 0.0
            accQ_odd = 0.0
            for nn in range(n_samples):
                base_i = mixI_loc[nn]
                base_q = mixQ_loc[nn]
                s = prn_2n[tau_start[tau] + nn]
                i = base_i if s else -base_i
                q = base_q if s else -base_q
                if nn % 2 == 0:
                    accI_even += i
                    accQ_even += q
                else:
                    accI_odd += i
                    accQ_odd += q
            accI_total = accI_even + accI_odd
            accQ_total = accQ_even + accQ_odd
            power = accI_total * accI_total + accQ_total * accQ_total
            corr_matrix[k, tau] = power

    t1 = time.perf_counter()

    row, col = np.unravel_index(int(np.argmax(corr_matrix)), corr_matrix.shape)
    peak = float(corr_matrix[row, col])
    mean_corr = float(np.mean(corr_matrix))
    peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
    return {
        "method": "cpu_fpga_compatible",
        "detected": int(peak_ratio >= ratio_min),
        "doppler": int(doppler_range[row]),
        "phase": int(col),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t1 - t0),
    }
def get_fpga_context(cfg):
    global FPGA_CONTEXT
    if FPGA_CONTEXT is not None:
        return FPGA_CONTEXT
    if not HAS_PYNQ:
        raise RuntimeError("pynq indisponible dans ce kernel")
    overlay = Overlay(cfg["bit_path"], download=True)
    FPGA_CONTEXT = {
        "overlay": overlay,
        "ip": getattr(overlay, "acquisition_serial_0"),
        "rx_dma": getattr(overlay, "axi_dma_0"),
        "prn_dma": getattr(overlay, "axi_dma_1"),
        "out_dma": getattr(overlay, "axi_dma_2"),
    }
    return FPGA_CONTEXT


def fpga_acquisition(signal, prn, cfg):
    ctx = get_fpga_context(cfg)
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])
    nb_fd = (fd_end - fd_start) // fd_step + 1
    total_outputs = nb_fd * nb_phases

    in_sig = allocate(shape=(n_samples,), dtype=np.int32)
    in_prn = allocate(shape=(n_samples,), dtype=np.int32)
    out_buf = allocate(shape=(total_outputs,), dtype=np.int32)

    in_sig[:] = np.rint(np.real(np.asarray(signal[:n_samples]))).astype(np.int32)
    in_prn[:] = np.rint(np.real(np.asarray(prn[:n_samples]))).astype(np.int32)
    out_buf[:] = 0

    try:
        try:
            ctx["ip"].register_map.fd_step = int(fd_step)
        except Exception:
            ctx["ip"].write(0x40, int(fd_step))

        t0 = time.perf_counter()
        ctx["out_dma"].recvchannel.transfer(out_buf)
        ctx["ip"].write(0x00, 0x01)
        ctx["rx_dma"].sendchannel.transfer(in_sig)
        ctx["prn_dma"].sendchannel.transfer(in_prn)

        start_wait = time.perf_counter()
        while True:
            ctrl = int(ctx["ip"].read(0x00))
            if ((ctrl >> 1) & 1) == 1:
                break
            if (time.perf_counter() - start_wait) > 20.0:
                raise TimeoutError("Timeout: ap_done non atteint")
            time.sleep(0.001)

        ctx["rx_dma"].sendchannel.wait()
        ctx["prn_dma"].sendchannel.wait()
        ctx["out_dma"].recvchannel.wait()
        t1 = time.perf_counter()

        arr = np.asarray(out_buf, dtype=np.float64)
        peak = float(np.max(arr))
        mean_corr = float(np.mean(arr))
        peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
        return {
            "method": "fpga",
            "detected": int(peak_ratio >= ratio_min),
            "doppler": int(to_s32(ctx["ip"].read(0x10))),
            "phase": int(ctx["ip"].read(0x20)),
            "peak": peak,
            "peak_ratio": peak_ratio,
            "time_s": float(t1 - t0),
        }
    finally:
        in_sig.freebuffer()
        in_prn.freebuffer()
        out_buf.freebuffer()


def run_method(method_name, signal, prn_seq, cfg):
    if method_name == "cpu_float":
        return cpu_float_acquisition(signal, prn_seq, cfg)
    if method_name == "cpu_fpga_compatible":
        return cpu_fpga_compatible_acquisition(signal, prn_seq, cfg)
    if method_name == "fpga":
        return fpga_acquisition(signal, prn_seq, cfg)
    raise ValueError(f"Méthode inconnue: {method_name}")


def build_winner_table(df_raw, cfg):
    winner_rows = []
    for (file_name, method_name), group in df_raw.groupby(["file", "method"], dropna=False):
        valid_group = group[group["status"] == "ok"].sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        if valid_group.empty:
            error_row = group.iloc[0]
            winner_rows.append({
                "file": file_name,
                "method": method_name,
                "status": error_row["status"],
                "error_message": error_row.get("error_message", ""),
            })
            continue

        top1 = valid_group.iloc[0]
        top2_ratio = float(valid_group.iloc[1]["peak_ratio"]) if len(valid_group) > 1 else np.nan
        winner_margin_db = float(10.0 * np.log10((float(top1["peak_ratio"]) + EPS) / (top2_ratio + EPS))) if not np.isnan(top2_ratio) else np.nan
        robust_detected = int(
            (float(top1["peak_ratio"]) >= float(cfg["detection_ratio_min"]))
            and (np.isnan(winner_margin_db) or winner_margin_db >= float(cfg["winner_margin_db_min"]))
        )

        doppler_err = int(abs(int(top1["doppler_meas_hz"]) - int(top1["doppler_true_hz"])))
        phase_err = int(phase_error_mod(int(top1["phase_meas_chip"]), int(top1["phase_true_chip"]), int(cfg["nb_phases"])))
        prn_ok = int(int(top1["prn_tested"]) == int(top1["prn_injected"]))
        doppler_ok = int(doppler_err <= int(cfg["doppler_tol_hz"]))
        phase_ok = int(phase_err <= int(cfg["phase_tol_chip"]))
        strict_success = int((robust_detected == 1) and (prn_ok == 1) and (doppler_ok == 1) and (phase_ok == 1))
        false_alarm = int((robust_detected == 1) and (prn_ok == 0))
        miss = int(robust_detected == 0)

        winner_rows.append({
            "file": file_name,
            "method": method_name,
            "status": "ok",
            "error_message": "",
            "snr_db": float(top1["snr_db"]),
            "prn_injected": int(top1["prn_injected"]),
            "prn_winner": int(top1["prn_tested"]),
            "doppler_true_hz": int(top1["doppler_true_hz"]),
            "doppler_meas_hz": int(top1["doppler_meas_hz"]),
            "doppler_err_hz": doppler_err,
            "phase_true_chip": int(top1["phase_true_chip"]),
            "phase_meas_chip": int(top1["phase_meas_chip"]),
            "phase_err_chip": phase_err,
            "peak": float(top1["peak"]),
            "peak_ratio": float(top1["peak_ratio"]),
            "second_peak_ratio": top2_ratio,
            "winner_margin_db": winner_margin_db,
            "time_s": float(top1["time_s"]),
            "classic_detected": int(top1["detected"]),
            "robust_detected": robust_detected,
            "pd_flag": robust_detected,
            "pfa_flag": false_alarm,
            "pm_flag": miss,
            "prn_ok": prn_ok,
            "doppler_ok": doppler_ok,
            "phase_ok": phase_ok,
            "strict_success": strict_success,
        })
    return pd.DataFrame(winner_rows)


cfg = get_current_config() if "get_current_config" in globals() else dict(CONFIG)
CONFIG = dict(cfg)

signals_dir = Path(cfg["signals_dir"])
prn_dir = Path(cfg["prn_dir"])

if not signals_dir.exists():
    raise FileNotFoundError(f"Dossier signaux introuvable: {signals_dir}")
if not prn_dir.exists():
    raise FileNotFoundError(f"Dossier PRN introuvable: {prn_dir}")
if int(cfg["fd_step"]) <= 0:
    raise ValueError("fd_step doit être strictement positif.")
if int(cfg["fd_end"]) < int(cfg["fd_start"]):
    raise ValueError("La borne Doppler de fin doit être >= à la borne de début.")

signal_files = collect_signal_files(signals_dir, cfg["signal_glob"])
if not signal_files:
    raise FileNotFoundError(f"Aucun signal avec métadonnées GPS trouvé dans {signals_dir}")

if cfg.get("limit_signals"):
    signal_files = signal_files[: int(cfg["limit_signals"])]

available_prns = get_available_prns(prn_dir)
if not available_prns:
    raise FileNotFoundError(f"Aucun fichier PRN trouvé dans {prn_dir}")

if cfg.get("prn_list"):
    selected_prns = [int(prn) for prn in cfg["prn_list"] if int(prn) in available_prns]
else:
    selected_prns = available_prns

if not selected_prns:
    raise ValueError("La liste des PRN à tester est vide.")

print("=== CAMPAGNE DE VALIDATION ===")
print(f"Nombre de signaux: {len(signal_files)}")
print(f"Nombre de PRN par signal: {len(selected_prns)}")
print(f"Méthodes: {[METHOD_LABELS.get(m, m) for m in cfg['run_methods']]}")
print(f"Plage Doppler: [{cfg['fd_start']}, {cfg['fd_end']}] Hz")
print(f"Pas Doppler fd_step: {cfg['fd_step']} Hz")
print()

rows = []
prn_cache = {}

for index_signal, sig_file in enumerate(signal_files, 1):
    meta = parse_meta_from_header(sig_file)
    prn_injected = int(meta["prn"])
    doppler_true_hz = int(meta["doppler"])
    phase_true_chip = int(meta["ca_shift"] % int(cfg["nb_phases"]))
    snr_db = float(meta["snr"])

    signal = load_signal_pm1(sig_file)
    n_samples = int(cfg["n_samples"])
    if signal.size < n_samples:
        signal = np.pad(signal, (0, n_samples - signal.size), mode="edge")
    else:
        signal = signal[:n_samples]
    signal = signal.astype(np.complex64)

    print(f"[{index_signal}/{len(signal_files)}] {sig_file.name} | PRN={prn_injected} | SNR={snr_db:g} dB")
    #     for prn_tested in selected_prns: # pour tester uniquement le prn present 
    for prn_tested in selected_prns:  # pour tester tous les prn pour chaque signal
        if prn_tested not in prn_cache:
            prn_cache[prn_tested] = load_prn_sequence(prn_dir, prn_tested, n_samples)
        prn_seq = prn_cache[prn_tested]

        for method_name in cfg["run_methods"]:
            try:
                result = run_method(method_name, signal, prn_seq, cfg)
                rows.append({
                    "file": sig_file.name,
                    "method": method_name,
                    "status": "ok",
                    "error_message": "",
                    "snr_db": snr_db,
                    "prn_injected": prn_injected,
                    "prn_tested": int(prn_tested),
                    "doppler_true_hz": doppler_true_hz,
                    "phase_true_chip": phase_true_chip,
                    "detected": int(result["detected"]),
                    "doppler_meas_hz": int(result["doppler"]),
                    "phase_meas_chip": int(result["phase"]),
                    "peak": float(result["peak"]),
                    "peak_ratio": float(result["peak_ratio"]),
                    "time_s": float(result["time_s"]),
                })
            except Exception as exc:
                rows.append({
                    "file": sig_file.name,
                    "method": method_name,
                    "status": "error",
                    "error_message": str(exc),
                    "snr_db": snr_db,
                    "prn_injected": prn_injected,
                    "prn_tested": int(prn_tested),
                    "doppler_true_hz": doppler_true_hz,
                    "phase_true_chip": phase_true_chip,
                    "detected": 0,
                    "doppler_meas_hz": np.nan,
                    "phase_meas_chip": np.nan,
                    "peak": np.nan,
                    "peak_ratio": np.nan,
                    "time_s": np.nan,
                })

raw_results = pd.DataFrame(rows)
if raw_results.empty:
    raise RuntimeError("Aucun résultat brut produit.")

winner_results = build_winner_table(raw_results, cfg)
if winner_results.empty:
    raise RuntimeError("Aucun résultat winner produit.")

domain_summary = pd.DataFrame([
    {
        "n_signals": len(signal_files),
        "signals_dir": str(signals_dir.resolve()),
        "prn_dir": str(prn_dir.resolve()),
        "methods": ", ".join(cfg["run_methods"]),
        "n_prn_tested_per_signal": len(selected_prns),
        "snr_min_db": float(min(parse_meta_from_header(p)["snr"] for p in signal_files)),
        "snr_max_db": float(max(parse_meta_from_header(p)["snr"] for p in signal_files)),
        "doppler_min_hz": int(min(parse_meta_from_header(p)["doppler"] for p in signal_files)),
        "doppler_max_hz": int(max(parse_meta_from_header(p)["doppler"] for p in signal_files)),
        "prn_min": int(min(parse_meta_from_header(p)["prn"] for p in signal_files)),
        "prn_max": int(max(parse_meta_from_header(p)["prn"] for p in signal_files)),
        "phase_min_chip": int(min(parse_meta_from_header(p)["ca_shift"] for p in signal_files)),
        "phase_max_chip": int(max(parse_meta_from_header(p)["ca_shift"] for p in signal_files)),
        "fs_hz": float(cfg["fs_hz"]),
        "if_hz": float(cfg["if_hz"]),
        "n_samples": int(cfg["n_samples"]),
        "fd_start": int(cfg["fd_start"]),
        "fd_end": int(cfg["fd_end"]),
        "fd_step": int(cfg["fd_step"]),
        "doppler_tol_hz": int(cfg["doppler_tol_hz"]),
        "phase_tol_chip": int(cfg["phase_tol_chip"]),
        "peak_ratio_min": float(cfg["detection_ratio_min"]),
        "winner_margin_db_min": float(cfg["winner_margin_db_min"]),
    }
])
export_dir = Path("Resultats validation SA")
export_dir.mkdir(exist_ok=True)
prefix = cfg["export_prefix"]
raw_results.to_csv(export_dir / f"{prefix}_raw_results.csv", index=False)
winner_results.to_csv(export_dir / f"{prefix}_winner_results.csv", index=False)
domain_summary.to_csv(export_dir / f"{prefix}_domain_summary.csv", index=False)

print("\n=== EXPORTS ===")
print(f"- {export_dir / f'{prefix}_raw_results.csv'}")
print(f"- {export_dir / f'{prefix}_winner_results.csv'}")
print(f"- {export_dir / f'{prefix}_domain_summary.csv'}")
print("\n=== APERCU DOMAINE DE VALIDITE ===")
display(domain_summary.T) if HAS_DISPLAY else print(domain_summary.T.to_string())
print("\n=== APERCU RESULTATS WINNER ===")
display(winner_results.head(10)) if HAS_DISPLAY else print(winner_results.head(10).to_string(index=False))

=== CAMPAGNE DE VALIDATION ===
Nombre de signaux: 86
Nombre de PRN par signal: 1
Méthodes: ['CPU float', 'CPU compatible FPGA', 'FPGA']
Plage Doppler: [-10000, 10000] Hz
Pas Doppler fd_step: 250 Hz

[1/86] gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv | PRN=25 | SNR=-10 dB


[2/86] gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n1250Hz_ca-804.csv | PRN=25 | SNR=-10 dB
[3/86] gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n2500Hz_ca-804.csv | PRN=25 | SNR=-10 dB
[4/86] gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n3750Hz_ca-804.csv | PRN=25 | SNR=-10 dB
[5/86] gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n5000Hz_ca-804.csv | PRN=25 | SNR=-10 dB


In [ ]:
# ==========================================================
# CELLULE 3 - TABLEAUX SCIENTIFIQUES PRINCIPAUX
# 1. Tableau global par méthode
# 2. Tableau par SNR
# 3. Domaine de validité rappelé
# ==========================================================
import numpy as np
import pandas as pd
from pathlib import Path
export_dir = Path("Resultats validation SA")
export_dir.mkdir(exist_ok=True)
try:
    from IPython.display import display
except Exception:
    pass

if "get_current_config" in dir():
    prefix = get_current_config()["export_prefix"]
elif "DEFAULT_CONFIG" in dir():
    prefix = DEFAULT_CONFIG["export_prefix"]
else:
    prefix = "validation_gnss"

_method_labels = METHOD_LABELS if "METHOD_LABELS" in dir() else {
    "cpu_float": "CPU float",
    "cpu_fpga_compatible": "CPU compatible FPGA",
    "fpga": "FPGA",
}

if "winner_results" not in globals() or not isinstance(winner_results, pd.DataFrame) or winner_results.empty:
    winner_results = pd.read_csv(f"{prefix}_winner_results.csv")
if "domain_summary" not in globals() or not isinstance(domain_summary, pd.DataFrame) or domain_summary.empty:
    domain_summary = pd.read_csv(f"{prefix}_domain_summary.csv")

ok = winner_results[winner_results["status"] == "ok"].copy()
if ok.empty:
    raise RuntimeError("Aucun résultat valide disponible pour l'analyse.")

# --- 1. Tableau global par méthode ---
method_global = (
    ok.groupby("method", as_index=False)
    .agg(
        n_signals=("file", "count"),
        strict_success_pct=("strict_success", lambda s: 100.0 * s.mean()),
        Pd_pct=("pd_flag", lambda s: 100.0 * s.mean()),
        Pfa_pct=("pfa_flag", lambda s: 100.0 * s.mean()),
        doppler_mae_hz=("doppler_err_hz", lambda s: np.mean(np.abs(s))),
        phase_mae_chip=("phase_err_chip", lambda s: np.mean(np.abs(s))),
        peak_ratio_mean=("peak_ratio", "mean"),
        time_mean_ms=("time_s", lambda s: 1000.0 * s.mean()),
        time_median_ms=("time_s", lambda s: 1000.0 * s.median()),
    )
    .sort_values("method")
    .reset_index(drop=True)
)
method_global["method_label"] = method_global["method"].map(_method_labels)

# --- 2. Tableau par SNR ---
snr_summary = (
    ok.groupby(["method", "snr_db"], as_index=False)
    .agg(
        n_signals=("file", "count"),
        strict_success_pct=("strict_success", lambda s: 100.0 * s.mean()),
        Pd_pct=("pd_flag", lambda s: 100.0 * s.mean()),
        Pfa_pct=("pfa_flag", lambda s: 100.0 * s.mean()),
        doppler_mae_hz=("doppler_err_hz", lambda s: np.mean(np.abs(s))),
        phase_mae_chip=("phase_err_chip", lambda s: np.mean(np.abs(s))),
        time_mean_ms=("time_s", lambda s: 1000.0 * s.mean()),
    )
    .sort_values(["snr_db", "method"])
    .reset_index(drop=True)
)
snr_summary["method_label"] = snr_summary["method"].map(_method_labels)

# --- Exports ---
method_global.to_csv(export_dir / f"{prefix}_table_global.csv", index=False)
snr_summary.to_csv(export_dir / f"{prefix}_table_by_snr.csv", index=False)

print("=== TABLEAU GLOBAL PAR METHODE ===")
display(method_global.round(4))

print("\n=== TABLEAU PAR SNR ===")
display(snr_summary.round(4))

if not domain_summary.empty:
    print("\n=== DOMAINE DE VALIDITE ===")
    display(domain_summary)

print("\nExports:")
print(f"- {export_dir / f'{prefix}_table_global.csv'}")
print(f"- {export_dir / f'{prefix}_table_by_snr.csv'}")

In [ ]:
# ==========================================================
# CELLULE 4 - CONCORDANCE CPU / FPGA ET COUT DE CALCUL
# 1. Tableau CPU vs FPGA
# 2. Accord de décision et d'estimation entre méthodes
# ==========================================================
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    pass

if "get_current_config" in dir():
    prefix = get_current_config()["export_prefix"]
elif "DEFAULT_CONFIG" in dir():
    prefix = DEFAULT_CONFIG["export_prefix"]
else:
    prefix = "validation_gnss"

_method_labels = METHOD_LABELS if "METHOD_LABELS" in dir() else {
    "cpu_float": "CPU float",
    "cpu_fpga_compatible": "CPU compatible FPGA",
    "fpga": "FPGA",
}

if "winner_results" not in globals() or not isinstance(winner_results, pd.DataFrame) or winner_results.empty:
    winner_results = pd.read_csv(f"{prefix}_winner_results.csv")

ok = winner_results[winner_results["status"] == "ok"].copy()
if ok.empty:
    raise RuntimeError("Aucun résultat valide disponible.")

base_runtime = (
    ok.groupby("method", as_index=False)["time_s"]
    .mean()
    .rename(columns={"time_s": "time_mean_s"})
)
if "cpu_float" in set(base_runtime["method"]):
    cpu_ref = float(base_runtime.loc[base_runtime["method"] == "cpu_float", "time_mean_s"].iloc[0])
else:
    cpu_ref = float(base_runtime["time_mean_s"].min())

cpu_fpga_table = (
    ok.groupby("method", as_index=False)
    .agg(
        n_signals=("file", "count"),
        success_pct=("strict_success", lambda s: 100.0 * s.mean()),
        Pd_pct=("pd_flag", lambda s: 100.0 * s.mean()),
        Pfa_pct=("pfa_flag", lambda s: 100.0 * s.mean()),
        doppler_mae_hz=("doppler_err_hz", lambda s: np.mean(np.abs(s))),
        phase_mae_chip=("phase_err_chip", lambda s: np.mean(np.abs(s))),
        peak_ratio_mean=("peak_ratio", "mean"),
        time_mean_ms=("time_s", lambda s: 1000.0 * s.mean()),
        time_median_ms=("time_s", lambda s: 1000.0 * s.median()),
        time_p95_ms=("time_s", lambda s: 1000.0 * np.nanpercentile(s, 95)),
    )
    .sort_values("method")
    .reset_index(drop=True)
)
cpu_fpga_table["speedup_vs_cpu_float"] = cpu_ref / (cpu_fpga_table["time_mean_ms"] / 1000.0)
cpu_fpga_table["method_label"] = cpu_fpga_table["method"].map(_method_labels).fillna(cpu_fpga_table["method"])

agreement_rows = []
methods = sorted(ok["method"].unique())
wide = ok.pivot(index="file", columns="method", values=["strict_success", "prn_winner", "doppler_meas_hz", "phase_meas_chip"])
for idx_left, method_left in enumerate(methods):
    for method_right in methods[idx_left + 1:]:
        cols = [
            ("strict_success", method_left),
            ("strict_success", method_right),
            ("prn_winner", method_left),
            ("prn_winner", method_right),
            ("doppler_meas_hz", method_left),
            ("doppler_meas_hz", method_right),
            ("phase_meas_chip", method_left),
            ("phase_meas_chip", method_right),
        ]
        common = wide[cols].dropna()
        if common.empty:
            continue
        agreement_rows.append({
            "method_left": method_left,
            "method_right": method_right,
            "method_left_label": _method_labels.get(method_left, method_left),
            "method_right_label": _method_labels.get(method_right, method_right),
            "n_common": int(len(common)),
            "decision_agreement_pct": float(100.0 * np.mean(common[("strict_success", method_left)] == common[("strict_success", method_right)])),
            "prn_agreement_pct": float(100.0 * np.mean(common[("prn_winner", method_left)] == common[("prn_winner", method_right)])),
            "doppler_delta_mean_hz": float(np.mean(np.abs(common[("doppler_meas_hz", method_left)] - common[("doppler_meas_hz", method_right)]))),
            "phase_delta_mean_chip": float(np.mean(np.abs(common[("phase_meas_chip", method_left)] - common[("phase_meas_chip", method_right)]))),
        })

agreement_table = pd.DataFrame(agreement_rows)

cpu_fpga_table.to_csv(export_dir / f"{prefix}_table_cpu_fpga.csv", index=False)
agreement_table.to_csv(export_dir / f"{prefix}_table_method_agreement.csv", index=False)

print("=== TABLEAU CPU VS FPGA ===")
display(cpu_fpga_table.round(4))

print("\n=== CONCORDANCE ENTRE METHODES ===")
if agreement_table.empty:
    print("Une seule méthode disponible: pas de tableau de concordance multi-méthodes.")
else:
    display(agreement_table.round(4))

print("\nExports:")
print(f"- {export_dir / f'{prefix}_table_cpu_fpga.csv'}")
print(f"- {export_dir / f'{prefix}_table_method_agreement.csv'}")

# NOTEBOOK DE VALIDATION SCIENTIFIQUE GNSS

Ce notebook est organisé pour répondre à trois questions de validation:

1. L'algorithme détecte-t-il correctement le satellite?
2. L'algorithme estime-t-il correctement Doppler et phase de code?
3. L'algorithme reste-t-il robuste quand le SNR baisse et quand on compare CPU vs FPGA?

Ordre recommandé d'exécution:

1. Cellule 1: configuration de la campagne, critères de validation, chemins de données.
2. Cellule 2: campagne de mesure CPU et FPGA sur les signaux générés.
3. Cellule 3: tableaux scientifiques principaux.
4. Cellule 4: tableau CPU vs FPGA et concordance entre méthodes.
5. Cellule 5: cas difficiles et graphes pour le rapport.
6. Cellule 6: graphes séparés par méthode.

Critère de succès strict utilisé dans ce notebook:

- détection robuste valide
- PRN gagnant correct
- erreur Doppler dans la tolérance
- erreur de phase dans la tolérance

Tableaux minimum produits:

- tableau global par méthode
- tableau par SNR
- tableau CPU vs FPGA
- tableau de concordance entre méthodes
- tableau des cas difficiles
- tableau du domaine de validité testé

Exports principaux:

- validation_gnss_raw_results.csv
- validation_gnss_winner_results.csv
- validation_gnss_domain_summary.csv
- validation_gnss_table_global.csv
- validation_gnss_table_by_snr.csv
- validation_gnss_table_cpu_fpga.csv
- validation_gnss_table_method_agreement.csv
- validation_gnss_table_hard_cases.csv

## Glossaire des abreviations

- Pd: Probabilite de detection (plus c'est proche de 100%, mieux c'est)
- Pfa: Probabilite de fausse alarme (plus c'est proche de 0%, mieux c'est)
- PM: Probabilite de non-detection (missed detection)
- MAE: Mean Absolute Error, erreur absolue moyenne
- SNR: Signal-to-Noise Ratio, rapport signal sur bruit (en dB)
- PRN: Pseudo-Random Noise code, identifiant de code satellite GPS
- CPU: Central Processing Unit
- FPGA: Field-Programmable Gate Array
- IF: Intermediate Frequency, frequence intermediaire
- fs: Frequence d'echantillonnage
- fd: Frequence Doppler testee (offset Doppler)
- CA code: Coarse/Acquisition code GPS (code C/A)
- p95: 95e percentile (valeur en dessous de laquelle se trouvent 95% des mesures)
- strict_success: Succes strict (detection robuste + PRN correct + Doppler dans tolerance + phase dans tolerance)

In [ ]:
# ==========================================================
# CELLULE 5 - CAS DIFFICILES ET VISUELS DE VALIDATION
# 1. Tableau des pires cas
# 2. Courbes robustesse vs SNR
# 3. Temps de calcul par méthode
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    pass

if "get_current_config" in dir():
    _cfg = get_current_config()
elif "DEFAULT_CONFIG" in dir():
    _cfg = DEFAULT_CONFIG
else:
    _cfg = {"export_prefix": "validation_gnss", "doppler_tol_hz": 500, "phase_tol_chip": 3}
prefix = _cfg["export_prefix"]

_method_labels = METHOD_LABELS if "METHOD_LABELS" in dir() else {
    "cpu_float": "CPU float",
    "cpu_fpga_compatible": "CPU compatible FPGA",
    "fpga": "FPGA",
}

if "winner_results" not in globals() or not isinstance(winner_results, pd.DataFrame) or winner_results.empty:
    winner_results = pd.read_csv(f"{prefix}_winner_results.csv")
if "snr_summary" not in globals() or not isinstance(snr_summary, pd.DataFrame) or snr_summary.empty:
    snr_summary = pd.read_csv(f"{prefix}_table_by_snr.csv")
if "cpu_fpga_table" not in globals() or not isinstance(cpu_fpga_table, pd.DataFrame) or cpu_fpga_table.empty:
    cpu_fpga_table = pd.read_csv(f"{prefix}_table_cpu_fpga.csv")

ok = winner_results[winner_results["status"] == "ok"].copy()
if ok.empty:
    raise RuntimeError("Aucun résultat valide disponible.")

hard_cases = ok.sort_values(
    ["strict_success", "winner_margin_db", "peak_ratio", "doppler_err_hz", "phase_err_chip"],
    ascending=[True, True, True, False, False],
).reset_index(drop=True)

hard_cases = hard_cases[
    [
        "file",
        "method",
        "snr_db",
        "prn_injected",
        "prn_winner",
        "pd_flag",
        "pfa_flag",
        "pm_flag",
        "doppler_true_hz",
        "doppler_meas_hz",
        "doppler_err_hz",
        "phase_true_chip",
        "phase_meas_chip",
        "phase_err_chip",
        "peak_ratio",
        "winner_margin_db",
        "strict_success",
        "time_s",
    ]
].head(10)

hard_cases.to_csv(export_dir / f"{prefix}_table_hard_cases.csv", index=False)

print("=== TABLEAU DES CAS DIFFICILES ===")
display(hard_cases.round(4))

_has_pfa = "Pfa_pct" in snr_summary.columns

if not snr_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for method_name, group in snr_summary.groupby("method"):
        label = _method_labels.get(method_name, method_name)
        axes[0].plot(group["snr_db"], group["Pd_pct"], marker="o", label=label)
        if _has_pfa:
            axes[0].plot(group["snr_db"], group["Pfa_pct"], marker="x", linestyle="--", label=f"{label} Pfa")
        axes[1].plot(group["snr_db"], group["doppler_mae_hz"], marker="o", label=label)
        axes[2].plot(group["snr_db"], group["phase_mae_chip"], marker="o", label=label)

    axes[0].set_title("Detection et fausse alarme vs SNR")
    axes[0].set_xlabel("SNR (dB)")
    axes[0].set_ylabel("%")
    axes[0].set_ylim(-2, 102)
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].axhline(_cfg["doppler_tol_hz"], color="red", linestyle="--", alpha=0.7)
    axes[1].set_title("Erreur Doppler MAE vs SNR")
    axes[1].set_xlabel("SNR (dB)")
    axes[1].set_ylabel("Hz")
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)

    axes[2].axhline(_cfg["phase_tol_chip"], color="red", linestyle="--", alpha=0.7)
    axes[2].set_title("Erreur phase MAE vs SNR")
    axes[2].set_xlabel("SNR (dB)")
    axes[2].set_ylabel("chips")
    axes[2].grid(alpha=0.3)
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

if not cpu_fpga_table.empty:
    _method_col = "method_label" if "method_label" in cpu_fpga_table.columns else "method"
    plt.figure(figsize=(7.2, 4.2))
    plt.bar(cpu_fpga_table[_method_col], cpu_fpga_table["time_mean_ms"], color=["#4C78A8", "#72B7B2", "#F58518"][: len(cpu_fpga_table)])
    plt.title("Temps moyen par méthode")
    plt.ylabel("ms")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\nExport:")
print(f"- {export_dir / f'{prefix}_table_hard_cases.csv'}")

In [ ]:
# ==========================================================
# CELLULE 6 - GRAPHIQUES SEPARES PAR METHODE
# Affiche une figure distincte pour chaque méthode
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    pass

if "get_current_config" in dir():
    _cfg = get_current_config()
elif "DEFAULT_CONFIG" in dir():
    _cfg = DEFAULT_CONFIG
else:
    _cfg = {"export_prefix": "validation_gnss", "doppler_tol_hz": 500, "phase_tol_chip": 3}

prefix = _cfg["export_prefix"]

_method_labels = METHOD_LABELS if "METHOD_LABELS" in dir() else {
    "cpu_float": "CPU float",
    "cpu_fpga_compatible": "CPU compatible FPGA",
    "fpga": "FPGA",
}

if "snr_summary" not in globals() or not isinstance(snr_summary, pd.DataFrame) or snr_summary.empty:
    snr_summary = pd.read_csv(f"{prefix}_table_by_snr.csv")
if "winner_results" not in globals() or not isinstance(winner_results, pd.DataFrame) or winner_results.empty:
    winner_results = pd.read_csv(f"{prefix}_winner_results.csv")

required_cols = {"method", "snr_db", "Pd_pct", "doppler_mae_hz", "phase_mae_chip"}
missing_cols = sorted(list(required_cols - set(snr_summary.columns)))
if missing_cols:
    raise KeyError(f"Colonnes manquantes dans snr_summary: {missing_cols}")

_has_pfa = "Pfa_pct" in snr_summary.columns

runtime_by_snr = (
    winner_results[winner_results["status"] == "ok"]
    .groupby(["method", "snr_db"], as_index=False)["time_s"]
    .mean()
)
runtime_by_snr["time_mean_ms"] = 1000.0 * runtime_by_snr["time_s"]

methods = list(dict.fromkeys(snr_summary["method"].tolist()))
if not methods:
    raise RuntimeError("Aucune méthode trouvée dans snr_summary.")

for method_name in methods:
    method_label = _method_labels.get(method_name, method_name)
    g = snr_summary[snr_summary["method"] == method_name].sort_values("snr_db")
    g_time = runtime_by_snr[runtime_by_snr["method"] == method_name].sort_values("snr_db")

    if g.empty:
        continue

    fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))

    axes[0].plot(g["snr_db"], g["Pd_pct"], marker="o", linewidth=2, label="Pd")
    if _has_pfa:
        axes[0].plot(g["snr_db"], g["Pfa_pct"], marker="x", linestyle="--", linewidth=1.8, label="Pfa")
    axes[0].set_title(f"{method_label} - Detection")
    axes[0].set_xlabel("SNR (dB)")
    axes[0].set_ylabel("%")
    axes[0].set_ylim(-2, 102)
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].plot(g["snr_db"], g["doppler_mae_hz"], marker="o", linewidth=2, color="#2ca02c")
    axes[1].axhline(_cfg["doppler_tol_hz"], color="red", linestyle="--", alpha=0.7, label="Tolérance")
    axes[1].set_title(f"{method_label} - Doppler MAE")
    axes[1].set_xlabel("SNR (dB)")
    axes[1].set_ylabel("Hz")
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)

    axes[2].plot(g["snr_db"], g["phase_mae_chip"], marker="o", linewidth=2, color="#9467bd")
    axes[2].axhline(_cfg["phase_tol_chip"], color="red", linestyle="--", alpha=0.7, label="Tolérance")
    axes[2].set_title(f"{method_label} - Phase MAE")
    axes[2].set_xlabel("SNR (dB)")
    axes[2].set_ylabel("chips")
    axes[2].grid(alpha=0.3)
    axes[2].legend(fontsize=8)

    if not g_time.empty:
        axes[3].plot(g_time["snr_db"], g_time["time_mean_ms"], marker="o", linewidth=2, color="#ff7f0e")
    axes[3].set_title(f"{method_label} - Temps moyen")
    axes[3].set_xlabel("SNR (dB)")
    axes[3].set_ylabel("ms")
    axes[3].grid(alpha=0.3)

    fig.suptitle(f"Validation GNSS - {method_label}", fontsize=12)
    plt.tight_layout()
    plt.show()

print(f"Figures générées: {len(methods)} méthode(s).")

## Glossaire des abreviations

- Pd: Probabilite de detection (plus c'est proche de 100%, mieux c'est)
- Pfa: Probabilite de fausse alarme (plus c'est proche de 0%, mieux c'est)
- PM: Probabilite de non-detection (missed detection)
- MAE: Mean Absolute Error, erreur absolue moyenne
- SNR: Signal-to-Noise Ratio, rapport signal sur bruit (en dB)
- PRN: Pseudo-Random Noise code, identifiant de code satellite GPS
- CPU: Central Processing Unit
- FPGA: Field-Programmable Gate Array
- IF: Intermediate Frequency, frequence intermediaire
- fs: Frequence d'echantillonnage
- fd: Frequence Doppler testee (offset Doppler)
- CA code: Coarse/Acquisition code GPS (code C/A)
- p95: 95e percentile (valeur en dessous de laquelle se trouvent 95% des mesures)
- strict_success: Succes strict (detection robuste + PRN correct + Doppler dans tolerance + phase dans tolerance)

In [ ]:
# ==========================================================
# CELLULE 8 - VISUALISATIONS DES RÉSULTATS (GRAPHIQUES)
# Inspiré des figures de la thèse (e.g., Figure 32-46)
# ==========================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Charger les données exportées (ajuste les chemins si nécessaire)
try:
    winner_results = pd.read_csv("validation_gnss_winner_results.csv")
    snr_summary = pd.read_csv("validation_gnss_table_by_snr.csv")
    method_global = pd.read_csv("validation_gnss_table_global.csv")
except FileNotFoundError:
    print("Erreur : Fichiers CSV non trouvés. Exécute d'abord les cellules 2-4 pour les générer.")
    raise

# Filtrer les données valides
ok = winner_results[winner_results["status"] == "ok"]

# 1. Taux de succès strict vs SNR (similaire à Figure 32-33 de la thèse)
if not snr_summary.empty:
    fig1 = px.line(
        snr_summary,
        x="snr_db",
        y="strict_success_pct",
        color="method",
        title="Taux de succès strict vs SNR (par méthode)",
        labels={"snr_db": "SNR (dB)", "strict_success_pct": "Taux de succès (%)"}
    )
    fig1.show()

# 2. Faux positifs vs SNR (similaire à Figure 34)
if not snr_summary.empty:
    fig2 = px.line(
        snr_summary,
        x="snr_db",
        y="Pfa_pct",
        color="method",
        title="Taux de faux positifs vs SNR",
        labels={"snr_db": "SNR (dB)", "Pfa_pct": "Faux positifs (%)"}
    )
    fig2.show()

# 3. Histogramme des erreurs Doppler (similaire à des analyses MAE)
if not ok.empty:
    fig3 = px.histogram(
        ok[ok["doppler_err_hz"] <= 1000],  # Filtre outliers
        x="doppler_err_hz",
        color="method",
        title="Distribution des erreurs Doppler (Hz)",
        nbins=50
    )
    fig3.show()

# 4. Temps d'exécution moyen vs méthode (similaire à analyses de performance)
if not method_global.empty:
    fig4 = px.bar(
        method_global,
        x="method",
        y="time_mean_ms",
        title="Temps d'exécution moyen par méthode (ms)",
        labels={"time_mean_ms": "Temps (ms)"}
    )
    fig4.show()

# 5. Robustesse : Succès vs erreurs simulées (si tu ajoutes des tests SE)
# Exemple : Si tu as des données avec % erreurs bits, ajoute une courbe décroissante comme Figure 35

print("Graphes générés. Ajuste les filtres ou ajoute des données pour plus de détails.")